# 🔬 GIADA supplemento Task 15b — collo di bottiglia dei gate
Matrice 2×2 con stessa MLP, stessi input, inizializzazione e minibatch per seed: loss gate ×4/×16, con o senza loss sulla corrente analitica. Checkpoint 400/800 step sulla stessa traiettoria. La Task 15 originale resta immutata; la Task 16 embedded non viene eseguita qui.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys
from IPython.display import Javascript,display
WORK=Path('/kaggle/working/giada_roadmap_task15b');REPO=WORK/'giada'
assert not WORK.exists(),f'Workspace esistente: {WORK}. Avvia una sessione pulita.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_gate_bottleneck_confirmation.py').is_file(),'Revisione senza Task 15b.'
sys.path.insert(0,str(REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher.roadmap_gate_bottleneck_confirmation import GateBottleneckConfig,verified_task15_result,train_and_freeze,evaluate_frozen
print({'revision':REVISION})


## 📁 Input necessario
Aggiungi agli Input Kaggle `giada_roadmap_task15_current_architecture_comparison.zip` (il risultato **originale** della Task 15). Può essere montato come ZIP oppure come cartella estratta; il report è verificato per SHA-256. Non serve il dataset HayFlow multi-GB. Se il nome del Dataset Kaggle è inatteso, imposta `GIADA_TASK15_ARTIFACT` al percorso preciso.


In [ ]:
INPUT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK15_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():
 candidates += [p for p in INPUT.rglob('*.zip') if 'task15' in str(p).lower() or 'current-architecture' in str(p).lower() or 'current_architecture' in str(p).lower()]
 candidates += [p.parent for p in INPUT.rglob('final_report.json') if 'task15' in str(p).lower() or 'current_architecture' in str(p).lower()]
TASK15=None;source_contract=None
for candidate in dict.fromkeys(candidates):
 if not candidate.exists():continue
 try:source_contract=verified_task15_result(candidate);TASK15=candidate.resolve();break
 except (ValueError,KeyError,FileNotFoundError,OSError):continue
assert TASK15 is not None,'Risultato Task 15 esatto non trovato. Aggiungi lo ZIP agli Input Kaggle oppure imposta GIADA_TASK15_ARTIFACT.'
display({'source':str(TASK15),'verified':source_contract})


## 🧠 Training appaiato e freeze
Train 192 episodi seed15211, development 48 seed15229. Tre seed 17/29/43. Stessa architettura da 4.966 parametri e stessi 10 ingressi per quattro bracci. La selezione del seed usa **solo m RMSE development a step 800**; il checkpoint 400 è un confronto di budget sulla stessa traiettoria, non una nuova selezione. Sealed e controfattuali hanno seed nuovi e restano chiusi fino al freeze. Il tracker stampa 24 righe in totale.


In [ ]:
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import torch
assert torch.cuda.is_available(),'La Task 15b preregistrata richiede una GPU CUDA Kaggle.'
DEVICE='cuda';config=GateBottleneckConfig();config.validate()
OUTPUT=Path('/kaggle/working/artifacts/giada_roadmap_task15b_gate_bottleneck_confirmation')
assert not OUTPUT.exists(),f'Output esistente: {OUTPUT}. Avvia una sessione pulita.'
print({'gpu':torch.cuda.get_device_name(0),'arms':4,'seeds':config.seeds,'steps':config.steps},flush=True)
freeze=train_and_freeze(TASK15,OUTPUT,config,device=DEVICE)
display({'selected':{k:{'seed':v['seed'],'development_m_rmse':v['development_m_rmse']} for k,v in freeze['selected'].items()},'freeze_sha256':freeze['freeze_sha256'],'sealed_accessed':freeze['sealed_accessed']})
assert not freeze['sealed_accessed'] and not freeze['counterfactual_accessed']


## 🔒 Valutazione sigillata
48 episodi sealed seed15259 e 48 episodi controfattuali appaiati seed15359. Confronta m/h, V, corrente, rollout16ms e delta-I/V. I criteri di successo registrati richiedono una riduzione di m senza peggiorare significativamente corrente, rollout e risposta causale. Un risultato negativo è informativo e non autorizza modifiche retroattive alle soglie.


In [ ]:
final=evaluate_frozen(freeze,OUTPUT,config,device=DEVICE,code_revision=REVISION)
compact={name:{'m_rmse':row['m_rmse'],'h_rmse':row['h_rmse'],'voltage_rmse_mv':row['voltage_rmse_mv'],'current_rmse_ma_cm2':row['all_rmse_ma_cm2'],'rollout_16ms_v_rmse_mv':row['recursive_16ms'].get('voltage_rmse_mv'),'rollout_valid':row['recursive_16ms']['valid']} for name,row in final['sealed'].items()}
display({'valid':final['valid'],'sealed':compact,'decisions':final['registered_decisions'],'counterfactual_conditions':list(final['counterfactuals'])})
if not final['valid']:print('ATTENZIONE: report non valido; scarica lo ZIP diagnostico ma non interpretare il ranking.')


## 📦 Scarica lo ZIP
Metodo Blob/base64 concordato per Kaggle; nessun FileLink o pannello Files.


In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task15b_gate_bottleneck_confirmation','zip',OUTPUT.parent,OUTPUT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
